# <center> Deliverable 02

### Task 1 - File Ingestion and reguar expressions

In [1]:
from pathlib import Path

import re, jason

# Task 1.1 
def read_log_file(file_path):
    path = Path(file_path)

    try:
        content = path.read_text(encoding="utf-8")
        return content, "utf-8"

    except UnicodeDecodeError:
        content = path.read_text(encoding="iso-8859-1")
        return content, "iso-8859-1"


# Task 1.2

def log_parser(log):
    log_lines = log.split('\n')
    logs = []
    pattern = r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] \[(POS_\d+)\] item=(\w+) price=(\d+\.\d+) (\w+) \((\w+)\)"
    
    for log_line in log_lines:
        match = re.match(pattern, log_line)
        if match:
            log_dict = {
                "timestamp": match.group(1),
                "register_id": match.group(2),
                "item_name": match.group(3),
                "price": match.group(4),
                "currency": match.group(5),
                "status": match.group(6)
            }
            logs.append(log_dict)
    
    return logs


ModuleNotFoundError: No module named 'jason'

### Task 2 - Recursive utility functions

In [ ]:
# Task 2.1

def find_log_files(directory):
    directory = Path(directory)
    log_files = []

    for item in directory.iterdir():

        if item.is_dir():
            log_files.extend(find_log_files(item))

        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files


# Task 2.2

def find_active_registers(data) -> set[str]:
    active_registers = set()

    if isinstance(data, dict):

        if data.get("active") is True and isinstance(data.get("id"), str):
            active_registers.add(data["id"])

        for value in data.values():
            active_registers.update(find_active_registers(value))

    elif isinstance(data, list):

        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers


# Testing task 2.1

data_path = Path("data")

log_files = find_log_files(data_path)

print("Number of log files:", len(log_files))

for file in log_files:
    print(file)


# Testing task 2.2

config_path = data_path / "store_config.json"

with config_path.open("r", encoding="utf-8") as file:
    config = json.load(file)

active_registers = find_active_registers(config)

print("\nActive registers:", active_registers)

### Task 3 - Functions for cleaning and analysing data

In [ ]:
import numpy as np

# Task 3.1

def remove_price_outliers(logs):
    prices = [d.get("price") for d in logs if "price" in d]

    mean_price = sum(float(price) for price in prices) / len(prices)
    std_dev = (sum((float(price) - mean_price) ** 2 for price in prices) / len(prices)) ** 0.5
    median_price = np.median([float(price) for price in prices])

    lower_bound = mean_price - 3 * std_dev
    upper_bound = mean_price + 3 * std_dev

    cleaned_logs = []

    for log in logs:
        if float(log["price"]) > lower_bound and float(log["price"]) < upper_bound:
            cleaned_logs.append(log)
        else:
            cleaned_logs.append(median_price)

    return np.array(cleaned_logs)

# Task 3.2

def compute_refund_ratio(logs):
    epsilon = 1e-6

    statuses = [d.get("status") for d in logs if "status" in d]

    sales_count = len([s for s in statuses if s == "SUCCESS"])
    refund_count = len([s for s in statuses if s == "REFUND"])

    refund_ratio = refund_count / (sales_count + epsilon)

    return refund_ratio


### Task 4 - Data analysis using Pandas

### Task 5 - Create full workflow